[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/traderbrasil/livro-dinheiro/blob/v2.0/notebooks/cap10_tesouro_direto.ipynb)

# Capítulo 10: Renda fixa I: Tesouro Direto

Laboratório do livro *Dinheiro: Querer, poder e gastar* (2ª edição), de Flávio Lemos, CMT, CFP®.

Você baixa o histórico oficial de preços e taxas do Tesouro Direto e vê, com dados reais, por que um título prefixado ou IPCA+ oscila antes do vencimento (a marcação a mercado).

**Como usar:** clique em cada célula e aperte o botão ▶ (ou Shift + Enter), de cima para baixo. Na primeira vez, o Colab pede para você entrar com uma conta do Google. Os campos marcados com `# MUDE AQUI` podem ser alterados com os seus números.

*Conteúdo educacional. Não é recomendação de investimento. Resultados passados não garantem resultados futuros. Os dados são baixados no dia em que você roda o notebook; por isso, os seus números podem ser diferentes dos do livro, que foram conferidos em outubro de 2026.*

In [ ]:
# Funções de apoio (não precisa mexer)
import io, time, math, datetime as dt
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

plt.rcParams.update({"figure.figsize": (10, 5), "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})
CORES = {"azul": "#1F3864", "verde": "#2E7D32", "laranja": "#E07B00", "vermelho": "#C62828",
         "cinza": "#757575", "azul_claro": "#5B9BD5", "roxo": "#6A1B9A"}

def brl(x, casas=2):
    """Formata um número como R$ 1.234,56."""
    s = f"{x:,.{casas}f}".replace(",", "X").replace(".", ",").replace("X", ".")
    return "R$ " + s

def pct(x, casas=2):
    return f"{x*100:,.{casas}f}%".replace(",", "X").replace(".", ",").replace("X", ".")

def eixo_reais(ax):
    ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda v, p: brl(v, 0)))

def sgs(codigo, inicio="01/01/2000", fim=None):
    """Baixa uma série do Sistema Gerenciador de Séries do Banco Central (SGS).
    Séries diárias são baixadas em blocos de 10 anos, o limite da API."""
    fim = fim or dt.date.today().strftime("%d/%m/%Y")
    d0 = dt.datetime.strptime(inicio, "%d/%m/%Y").date()
    d1 = dt.datetime.strptime(fim, "%d/%m/%Y").date()
    partes = []
    while d0 <= d1:
        d_fim = min(d1, d0.replace(year=d0.year + 9, month=12, day=31))
        url = (f"https://api.bcb.gov.br/dados/serie/bcdata.sgs.{codigo}/dados"
               f"?formato=json&dataInicial={d0:%d/%m/%Y}&dataFinal={d_fim:%d/%m/%Y}")
        for tentativa in range(4):
            try:
                r = requests.get(url, timeout=60, headers={"User-Agent": "Mozilla/5.0"})
                r.raise_for_status()
                dados = r.json()
                break
            except Exception:
                time.sleep(3 * (tentativa + 1))
        else:
            raise RuntimeError(f"Não consegui baixar a série {codigo}. Tente de novo em alguns minutos.")
        if dados:
            partes.append(pd.DataFrame(dados))
        d0 = d_fim + dt.timedelta(days=1)
    df = pd.concat(partes, ignore_index=True)
    df["data"] = pd.to_datetime(df["data"], dayfirst=True)
    df["valor"] = pd.to_numeric(df["valor"])
    return df.set_index("data")["valor"].sort_index()


## 1. Baixar os preços e as taxas do Tesouro Direto
O arquivo é publicado pelo Tesouro Nacional no portal Tesouro Transparente e tem todos os títulos desde 2002. Ele é grande (cerca de 15 MB); a primeira célula pode levar um minuto.

In [ ]:
URL_TD = ('https://www.tesourotransparente.gov.br/ckan/dataset/df56aa42-484a-4a59-8184-7676580c81e3/'
          'resource/796d2059-14e9-44e3-80c9-2d9e30b405c1/download/PrecoTaxaTesouroDireto.csv')
r = requests.get(URL_TD, timeout=180, headers={'User-Agent': 'Mozilla/5.0'})
td = pd.read_csv(io.BytesIO(r.content), sep=';', decimal=',')
td['Data Base'] = pd.to_datetime(td['Data Base'], dayfirst=True)
td['Data Vencimento'] = pd.to_datetime(td['Data Vencimento'], dayfirst=True)
print('Última data do arquivo:', td['Data Base'].max().strftime('%d/%m/%Y'))
hoje = td[td['Data Base'] == td['Data Base'].max()].sort_values(['Tipo Titulo', 'Data Vencimento'])
hoje[['Tipo Titulo', 'Data Vencimento', 'Taxa Compra Manha', 'PU Compra Manha']]

## 2. A taxa do IPCA+ longo ao longo do tempo
A taxa é o juro acima da inflação que o Tesouro paga a quem compra naquele dia.

In [ ]:
TITULO = 'Tesouro IPCA+'      # MUDE AQUI: 'Tesouro Prefixado', 'Tesouro IPCA+', 'Tesouro Selic'...
sel = td[td['Tipo Titulo'] == TITULO]
print('Vencimentos disponíveis hoje:', ', '.join(sorted({f'{v:%d/%m/%Y}' for v in sel[sel['Data Base'] == sel['Data Base'].max()]['Data Vencimento']})))
VENCIMENTO = pd.Timestamp('2035-05-15')   # MUDE AQUI: escolha um vencimento da lista acima (o de 2035 tem histórico desde 2010)
s = sel[sel['Data Vencimento'] == VENCIMENTO].set_index('Data Base').sort_index()
ax = s['Taxa Compra Manha'].plot(color=CORES['azul'], lw=1.5, title=f'{TITULO} {VENCIMENTO:%Y}: taxa de compra (% ao ano)')
ax.set_xlabel(''); plt.show()

## 3. A marcação a mercado
Quando as taxas sobem, o preço dos títulos já emitidos cai, e vice-versa. Quem leva até o vencimento recebe a taxa contratada; quem vende antes recebe o preço do dia.

In [ ]:
pu = s['PU Base Manha']
ax = pu.plot(color=CORES['laranja'], lw=1.5, title=f'{TITULO} {VENCIMENTO:%Y}: preço unitário (R$)')
eixo_reais(ax); ax.set_xlabel(''); plt.show()
var = pu.pct_change(21).dropna()   # cerca de um mês de pregões
print('Pior variação de preço em cerca de um mês:', pct(var.min()), 'em', var.idxmin().strftime('%m/%Y'))
print('Melhor variação em cerca de um mês:', pct(var.max()), 'em', var.idxmax().strftime('%m/%Y'))

## 4. Compare com o Tesouro Selic
O Tesouro Selic quase não oscila: por isso é o título indicado para a reserva de emergência (capítulo 8).

In [ ]:
ts = td[td['Tipo Titulo'] == 'Tesouro Selic']
v_selic = ts['Data Vencimento'].max()
ps = ts[ts['Data Vencimento'] == v_selic].set_index('Data Base').sort_index()['PU Base Manha']
vs = ps.pct_change(21).dropna()
print(f'Tesouro Selic {v_selic:%Y}: pior variação em cerca de um mês: {pct(vs.min())}')

Compare os títulos para um objetivo seu no artigo [Tesouro Direto: qual título escolher](https://www.traderbrasil.com/blog/tesouro-direto-selic-prefixado-ipca-qual-escolher.php).

---
**Hub do livro:** https://www.traderbrasil.com/livro-dinheiro/ · **Capítulo 10 no hub:** https://www.traderbrasil.com/livro-dinheiro/cap10/

Achou um erro? Escreva para contato@traderbrasil.com.